In [ ]:
import import_ipynb
import pandas as pd
from Data_Loader import DataLoader
from Feature_Loader import ConFeatures
from Models.ipynb import ConsumptionModel
from Plotting_Utils import Visualizer
from sklearn.metrics import r2_score, mean_absolute_error
from plotly.subplots import make_subplots
import plotly.graph_objects as go

loader = DataLoader()
viz = Visualizer()

ImportError: cannot import name 'ConsumptionModel' from 'Models' (unknown location)

In [ ]:
solcast, deop, _ = loader.load_training_data()
deop_2022, solcast_2022, _ = loader.load_testing_data()

viz.plot_monthly_box(deop, 'power-con-ave', "Monthly Consumption Distribution (2023)", True).show()

viz.plot_seasonal_profiles(deop, 'power-con-ave', "Seasonal Consumption Trends").show()

In [ ]:
viz.plot_monthly_box(deop_2022, 'power-con-ave', "Monthly Consumption Distribution (2022)", True).show()

viz.plot_seasonal_profiles(deop_2022, 'power-con-ave', "Seasonal Consumption Trends").show()

In [ ]:
con_features = [
            'air_temp', 'heating_demand', 'cooling_demand',
            'hour', 'day_of_week', 'month', 'is_holiday', 'non_working_day', 'non_working_hour_interaction',
            'is_term_time', 
            'hour_sin', 'hour_cos', 'day_sin', 'day_cos',
            'smart_lag', 'load_lag_7d', 
            'yesterday_daily_mean', 'yesterday_daily_max', 'rolling_7d_baseline'
        ]

con = ConsumptionModel(features=con_features)

con_preds, con_model = con.train_and_test(deop, solcast, deop_2022, solcast_2022, days_ahead=1)

In [ ]:
viz.plot_all(
    dfs=[deop_2022['power-con-ave'], con_preds],
    y_label='Consumption (kW)',
    data_labels=['Measured 2022', 'XGBoost Forecast'],
    title='Building Load: Prediction vs Actual'
).show()

In [ ]:
monthly_r2 = Visualizer.calculate_monthly_r2(deop_2022['power-con-ave'], con_preds, '2022-04-18 00:00:00')

In [ ]:

con_predictions = {}

for i in range(1, 15):
    preds, _ = con.train_and_test(
        deop, solcast, deop_2022, solcast_2022, days_ahead=i
    )
    
    con_predictions[f'{i}-Day Forecast'] = preds

In [ ]:
viz.plot_model_stats(
    actuals=deop_2022['power-con-ave'], 
    predictions=con_predictions,
    title='Consumption Model Performance'
).show()